# Indic Voice Pipeline — ASR evaluation on Colab GPU

Runs the full evaluation harness on a T4: base Whisper vs your LoRA adapter, on
identical audio, through the explicit `ASRRunner` (no `generate()`, no
`pipeline("asr")`).

**Runtime → Change runtime type → T4 GPU** before running anything.

What this notebook produces, per run:

| File | Contents |
| --- | --- |
| `run_config.json` | git SHA, versions, device, the exact sampled indices |
| `predictions.jsonl` | reference, hypothesis, duration, latency breakdown |
| `metrics.json` | WER/CER at 4 normalization levels, error categories, latency |
| `errors.jsonl` | per-example categorized edit operations |

Google Drive checkpoints are treated as **read-only** throughout.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi

## 2. Clone and install

`scripts/setup.sh` pins `datasets<4`. This is load-bearing: `google/fleurs` is
still a script-backed dataset, and `datasets>=4.0` removed loading-script
support entirely, so an unpinned install fails on every FLEURS load.

In [ ]:
!git clone https://github.com/Vaibhav7711/indic-voice-pipeline.git
%cd indic-voice-pipeline
!bash scripts/setup.sh

If `datasets` was already imported before the pin took effect,
**Runtime → Restart session** and re-run from the cell below (not the install).

In [ ]:
%cd /content/indic-voice-pipeline
!python scripts/preflight.py

## 3. Sanity-check the scoring layer

These tests are CPU-only and take under a second. They verify the metric
against an independent Levenshtein implementation and check the Devanagari
normalization rules. Run them before spending GPU time.

In [ ]:
!python -m pytest tests/test_metrics.py tests/test_normalize.py \
    tests/test_error_analysis.py tests/test_asr_eval.py tests/test_compare.py -q

## 4. Mount Drive for the checkpoint

Read-only. Nothing in this notebook writes to `whisper-training/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', readonly=True)

import pathlib
ADAPTER = "/content/drive/MyDrive/whisper-training/checkpoint-1400"

p = pathlib.Path(ADAPTER)
print("adapter dir exists:", p.is_dir())
if p.is_dir():
    for f in sorted(x.name for x in p.iterdir()):
        print("  ", f)

`loader.py` already handles the intermediate-checkpoint case: `checkpoint-*`
folders contain `adapter_config.json` and `adapter_model.safetensors` but often
no tokenizer files, so the processor is loaded from the base model unless
`tokenizer_config.json` is present in the adapter folder.

## 5. Configuration

`SEED` and `LIMIT` must be identical across both runs or the comparison is
meaningless — `benchmarks/compare.py` will refuse to report a delta otherwise.

Start with `LIMIT = 100` to confirm the pipeline end to end, then raise it.
FLEURS Hindi test is roughly 400 examples. To evaluate the full split, delete the `--limit {LIMIT}` flag from the run cells rather than setting `LIMIT = None` — shell interpolation would pass the literal string `None` to argparse.

In [ ]:
MODEL   = "openai/whisper-medium"
ADAPTER = "/content/drive/MyDrive/whisper-training/checkpoint-1400"
SPLIT   = "test"        # 'validation' for model selection; 'test' for final reporting only
LIMIT   = 100
SEED    = 0
LEVEL   = "standard"

BASE_DIR = f"results/eval/medium-base-{SPLIT}-{LIMIT}"
LORA_DIR = f"results/eval/medium-lora1400-{SPLIT}-{LIMIT}"
print(BASE_DIR, "\n", LORA_DIR)

## 6. Baseline: whisper-medium, no adapter

The first FLEURS load downloads and prepares the split — slow once, cached
after. `--warmup 2` discards two GPU passes before timing, because the first
forward pays for cuDNN autotuning and allocator growth and would otherwise
poison the mean RTF.

In [ ]:
!python -m benchmarks.asr_eval run \
    --model {MODEL} \
    --split {SPLIT} --limit {LIMIT} --seed {SEED} \
    --level {LEVEL} --warmup 2 \
    --out-dir {BASE_DIR}

## 7. Candidate: same audio, with the LoRA adapter merged

In [ ]:
!python -m benchmarks.asr_eval run \
    --model {MODEL} --adapter {ADAPTER} \
    --split {SPLIT} --limit {LIMIT} --seed {SEED} \
    --level {LEVEL} --warmup 2 \
    --out-dir {LORA_DIR}

## 8. Compare

This refuses to print a delta unless both runs used the same split, the same
example indices and the same normalization level. That guard is the point —
comparing a fine-tuned model on one sample against a baseline on another is the
easiest way to manufacture an improvement that is not real.

In [ ]:
!python -m benchmarks.compare \
    --baseline {BASE_DIR} \
    --candidate {LORA_DIR} \
    --out results/eval/compare-base-vs-lora1400.json

## 9. Read the failure modes

The error-category table says *what to fix next*:

| Dominant category | What it points at |
| --- | --- |
| `truncation` / `hallucination` | Decoding: `max_new_tokens`, early EOS, repetition |
| `orthographic` | Not a modelling error — a normalization policy decision |
| `code_switch` / `rare_word` | Training data: Hinglish, named entities |
| `numeric` | Output-format policy (digits vs words) |
| `function_word` | Usually audio quality, not vocabulary |

In [ ]:
import json
m = json.load(open(f"{LORA_DIR}/metrics.json"))

print("WER %", m["headline"]["wer_percent"], " CER %", m["headline"]["cer_percent"])
print("\nNormalization sensitivity:", json.dumps(m["normalization_sensitivity"], indent=2))
print("\nError categories:")
for name, payload in m["error_analysis"]["by_category"].items():
    print(f"  {name:<16}{payload['errors']:>6}{payload['share_of_errors_percent']:>8}%")
print("\nFlags:", m["error_analysis"]["flag_counts"])

### Worst examples — read these before changing anything

In [ ]:
for ex in m["error_analysis"]["worst_examples"][:8]:
    print(f"[{ex['id']}] WER {ex['wer']:.2f}")
    print("  ref:", ex["reference"])
    print("  hyp:", ex["hypothesis"])
    print()

### Top confusion pairs per category

In [ ]:
for name, payload in m["error_analysis"]["by_category"].items():
    pairs = payload["top_confusions"][:5]
    if pairs:
        print(f"{name}:")
        for pair, count in pairs:
            print(f"   {count:>3}x  {pair}")

## 10. Re-score without the GPU

Scoring has no torch dependency, so you can change the normalization level or
the structural-run threshold without re-running inference. Useful for asking
"how much of this WER is spelling convention?"

In [ ]:
!python -m benchmarks.asr_eval score \
    --predictions {LORA_DIR}/predictions.jsonl \
    --level aggressive \
    --out-dir {LORA_DIR}-aggressive

## 11. Seed the hard set from real failures

This writes `status="candidate"` items only. They are excluded from every
reported metric until you listen to each clip, correct the transcript, assign
categories, and flip the status to `curated`. See `data/hard_set/README.md`.

Some FLEURS references are simply wrong — those are bad references, not hard
audio. Delete them during review rather than promoting them.

In [ ]:
!python -m benchmarks.hard_set bootstrap \
    --predictions {LORA_DIR}/predictions.jsonl \
    --out data/hard_set/candidates.jsonl \
    --limit 30 --min-wer 0.4

!head -3 data/hard_set/candidates.jsonl

## 12. Save results off the ephemeral runtime

Copies evaluation outputs to Drive under a separate folder. Nothing touches
`whisper-training/`.

In [ ]:
import shutil, pathlib, datetime

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
dest = pathlib.Path(f"/content/drive/MyDrive/indic-voice-eval/{stamp}")

# Drive was mounted read-only above; remount writable only for this step.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

dest.mkdir(parents=True, exist_ok=True)
shutil.copytree("results/eval", dest / "eval", dirs_exist_ok=True)
print("saved to", dest)

## 13. Record the result

Open `docs/EXPERIMENTS.md` and fill in the result-entry template with the
numbers above. Leave any field blank rather than filling it from a different
run — a partially completed entry is evidence, a plausible-looking one is not.

`results/eval/` is gitignored by default. To version the run that backs a
recorded number:

```bash
git add -f results/eval/<dir>/metrics.json results/eval/<dir>/run_config.json
```